# Top 10 Conditions, Procedures, and Medications

Analysis of the Texas Workers' Compensation data using OMOP CDM tables.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect('/workspaces/txwc/tx_workers_comp.db', read_only=True)

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Top 10 Conditions

From the OMOP condition_occurrence table.

In [3]:
top_conditions = con.execute("""
    select 
        co.condition_source_value as source_code,
        c.concept_name as condition_name,
        count(*) as occurrence_count
    from omop.condition_occurrence co
    left join omop.concept c 
        on co.condition_concept_id = c.concept_id
    where co.condition_source_value is not null
    group by co.condition_source_value, c.concept_name
    order by occurrence_count desc
    limit 10
""").fetchdf()

print("Top 10 Conditions")
print("=" * 80)
top_conditions

Top 10 Conditions


,source_code,condition_name,occurrence_count
0,847.2,Lumbar sprain,2233616
1,724.2,Low back pain,1226121
2,847.0,Neck sprain,1021180
3,724.4,Peripheral neuritis,935635
4,840.9,Dislocations/sprains/strains,934531
5,722.10,Displacement of lumbar intervertebral disc without myelopathy,928007
6,S39.012A,"Injury of muscle and tendon of abdomen, lower back and pelvis",918443
7,S33.5XXD,Lumbar sprain,807208
8,719.41,Shoulder joint pain,741725
9,844.9,Soft tissue injury,702301


## Top 10 Procedures

From the OMOP procedure_occurrence table.

In [4]:
top_procedures = con.execute("""
    select 
        po.procedure_source_value as source_code,
        c.concept_name as procedure_name,
        count(*) as occurrence_count
    from omop.procedure_occurrence po
    left join omop.concept c 
        on po.procedure_concept_id = c.concept_id
    where po.procedure_source_value is not null
    group by po.procedure_source_value, c.concept_name
    order by occurrence_count desc
    limit 10
""").fetchdf()

print("Top 10 Procedures")
print("=" * 80)
top_procedures

Top 10 Procedures


,source_code,procedure_name,occurrence_count
0,G0283,"Electrical stimulation (unattended), to one or more areas for indication(s) other than wound care, as part of a therapy plan of care",1311879
1,99.99,Other miscellaneous procedures,350059
2,J3490,Unclassified drugs,100350
3,J8499,"Prescription drug, oral, non chemotherapeutic, nos",71430
4,G0299,"Direct skilled nursing services of a registered nurse (rn) in the home health or hospice setting, each 15 minutes",60649
5,S9124,"Nursing care, in the home; by licensed practical nurse, per hour",48978
6,J3590,Unclassified biologics,47378
7,S9123,"Nursing care, in the home; by registered nurse, per hour (use for general nursing care only, not to be used when cpt codes 99500-99602 can be used)",44500
8,J7799,"Noc drugs, other than inhalation drugs, administered through dme",37314
9,J7999,"Compounded drug, not otherwise classified",16532


## Top 10 Medications (by Drug Product)

From the OMOP drug_exposure table.

In [5]:
top_drugs = con.execute("""
    select 
        de.drug_source_value as ndc_code,
        c.concept_name as drug_name,
        count(*) as rx_count
    from omop.drug_exposure de
    left join omop.concept c 
        on de.drug_concept_id = c.concept_id
    where de.drug_source_value is not null
    group by de.drug_source_value, c.concept_name
    order by rx_count desc
    limit 10
""").fetchdf()

print("Top 10 Drug Products")
print("=" * 80)
top_drugs

Top 10 Drug Products


,ndc_code,drug_name,rx_count
0,J1885,ketorolac Injectable Solution [Toradol],358078
1,99999999999,No matching concept,275878
2,65162062711,tramadol hydrochloride 50 MG Oral Tablet,257623
3,J2405,ondansetron Injection,230114
4,J1100,dexamethasone Injection,210224
5,J3010,fentanyl 0.05 MG/ML Injection [Sublimaze],209299
6,53746046605,ibuprofen 800 MG Oral Tablet,179690
7,J2250,midazolam Injectable Solution [Versed],177339
8,J0690,cefazolin Injection,175098
9,00591085305,acetaminophen 325 MG / hydrocodone bitartrate 10 MG Oral Tablet,151498


## Top 10 Medications by Ingredient

Rolling up drug exposures to their RxNorm ingredient using concept_ancestor.

In [6]:
top_ingredients = con.execute("""
    select 
        ingredient.concept_name as ingredient_name,
        count(*) as rx_count
    from omop.drug_exposure de
    join omop.concept_ancestor ca 
        on de.drug_concept_id = ca.descendant_concept_id
    join omop.concept ingredient 
        on ca.ancestor_concept_id = ingredient.concept_id
    where ingredient.vocabulary_id = 'RxNorm'
      and ingredient.concept_class_id = 'Ingredient'
      and de.drug_concept_id is not null
      and de.drug_concept_id != 0
    group by ingredient.concept_name
    order by rx_count desc
    limit 10
""").fetchdf()

print("Top 10 Medication Ingredients")
print("=" * 80)
top_ingredients

Top 10 Medication Ingredients


,ingredient_name,rx_count
0,acetaminophen,2735379
1,hydrocodone,2052047
2,tramadol,940839
3,cyclobenzaprine,929015
4,ibuprofen,780152
5,gabapentin,748233
6,naproxen,665036
7,meloxicam,501974
8,pregabalin,404941
9,ketorolac,369808


## Top 10 Drug Classes by Established Pharmacologic Class (EPC)

Using the rxclass reference table to analyze drug exposures by FDA Established Pharmacologic Class.

In [7]:
top_epc = con.execute("""
    select 
        rc.className as drug_class,
        count(*) as rx_count
    from omop.drug_exposure de
    join omop.concept c 
        on de.drug_concept_id = c.concept_id
    join reference_data.rxclass rc 
        on c.concept_code = rc.rxcui
    where de.drug_concept_id is not null
      and de.drug_concept_id != 0
      and rc.classType = 'EPC'
    group by rc.className
    order by rx_count desc
    limit 10
""").fetchdf()

print("Top 10 Established Pharmacologic Classes (EPC)")
print("=" * 80)
top_epc

Top 10 Established Pharmacologic Classes (EPC)


,drug_class,rx_count
0,Opioid Agonist,6852984
1,Nonsteroidal Anti-inflammatory Drug,5467660
2,Muscle Relaxant,2817018
3,Corticosteroid,920426
4,Central alpha-2 Adrenergic Agonist,709478
5,Serotonin and Norepinephrine Reuptake Inhibitor,658322
6,Tricyclic Antidepressant,465730
7,gamma-Aminobutyric Acid A Receptor Positive Modulator,391790
8,Benzodiazepine,326216
9,Serotonin Reuptake Inhibitor,318958


## Top 10 Drug Classes by Disease Indication

Using the rxclass reference table to analyze drug exposures by disease-based classification.

In [8]:
top_disease = con.execute("""
    select 
        rc.className as disease_indication,
        count(*) as rx_count
    from omop.drug_exposure de
    join omop.concept c 
        on de.drug_concept_id = c.concept_id
    join reference_data.rxclass rc 
        on c.concept_code = rc.rxcui
    where de.drug_concept_id is not null
      and de.drug_concept_id != 0
      and rc.classType = 'DISEASE'
    group by rc.className
    order by rx_count desc
    limit 10
""").fetchdf()

print("Top 10 Disease Indications")
print("=" * 80)
top_disease

Top 10 Disease Indications


,disease_indication,rx_count
0,Drug Hypersensitivity,12727872
1,Pain,10023965
2,Fever,6808201
3,Asthma,4245702
4,Liver Diseases,3169662
5,"Arthritis, Rheumatoid",3082518
6,"Pain, Postoperative",3042634
7,Intestinal Obstruction,2950402
8,Respiratory Insufficiency,2822661
9,Osteoarthritis,2722828


## Summary Statistics

In [9]:
stats = con.execute("""
    select 'Conditions' as domain, count(*) as record_count from omop.condition_occurrence
    union all
    select 'Procedures', count(*) from omop.procedure_occurrence
    union all
    select 'Drug Exposures', count(*) from omop.drug_exposure
    union all
    select 'Measurements', count(*) from omop.measurement
    union all
    select 'Observations', count(*) from omop.observation
    union all
    select 'Visits', count(*) from omop.visit_occurrence
    union all
    select 'Persons', count(*) from omop.person
""").fetchdf()

print("OMOP Table Record Counts")
print("=" * 40)
stats

OMOP Table Record Counts


,domain,record_count
0,Conditions,84947725
1,Procedures,2605751
2,Drug Exposures,16789640
3,Measurements,1231275
4,Observations,10560423
5,Visits,50548780
6,Persons,24002149


---

# Enhanced Analysis: Cost, Benchmarking & Trends

The following sections integrate workers' compensation data with national Medicare benchmarks from Mimilabs.

In [10]:
# Setup for enhanced analysis - reopen connection
import requests
import json

# Reopen local database connection (was closed above)
con = duckdb.connect('/workspaces/txwc/tx_workers_comp.db', read_only=True)

# Helper function to query Mimilabs MCP API
MIMILABS_API = "https://www.mimilabs.ai/api/mcp"

def query_mimilabs(sql_query):
    """Execute a SQL query against Mimilabs healthcare data warehouse."""
    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {
            "name": "mimilabs_guide",
            "arguments": {"question": sql_query}
        }
    }
    response = requests.post(MIMILABS_API, json=payload, timeout=60)
    return response.json()

print("Enhanced analysis setup complete")

Enhanced analysis setup complete


## Drug Cost Analysis

Analyzing workers' compensation drug spending using the OMOP cost table.

In [11]:
# Top 10 drugs by total spending
drug_costs = con.execute("""
    SELECT 
        c.concept_name as drug_name,
        COUNT(*) as rx_count,
        ROUND(SUM(cost.total_charge), 2) as total_charges,
        ROUND(SUM(cost.total_paid), 2) as total_paid,
        ROUND(AVG(cost.total_paid), 2) as avg_paid_per_rx,
        ROUND(AVG(cost.total_charge), 2) as avg_charge_per_rx
    FROM omop.drug_exposure de
    JOIN omop.cost cost 
        ON de.drug_exposure_id = cost.cost_event_id 
        AND cost.cost_domain_id = 'Drug'
    LEFT JOIN omop.concept c 
        ON de.drug_concept_id = c.concept_id
    WHERE de.drug_concept_id IS NOT NULL 
      AND de.drug_concept_id != '0'
      AND cost.total_paid > 0
    GROUP BY c.concept_name
    ORDER BY total_paid DESC
    LIMIT 15
""").fetchdf()

print("Top 15 Drugs by Total Spending")
print("=" * 100)
drug_costs

Top 15 Drugs by Total Spending


,drug_name,rx_count,total_charges,total_paid,avg_paid_per_rx,avg_charge_per_rx
0,acetaminophen 325 MG / hydrocodone bitartrate 10 MG Oral Tablet,57699,10149628.95,8589409.41,148.87,175.91
1,tramadol hydrochloride 50 MG Oral Tablet,43303,7666537.11,6492675.46,149.94,177.04
2,cyclobenzaprine hydrochloride 10 MG Oral Tablet,38800,6813692.88,5814190.03,149.85,175.61
3,naproxen 500 MG Oral Tablet,32326,5688280.58,4811982.33,148.86,175.97
4,ibuprofen 800 MG Oral Tablet,27035,4787426.88,4077716.69,150.83,177.08
5,ketorolac Injectable Solution [Toradol],20525,3690674.07,3126170.91,152.31,179.81
6,gabapentin 300 MG Oral Capsule,18622,3294824.72,2787230.98,149.67,176.93
7,meloxicam 15 MG Oral Tablet,18266,3278671.59,2779456.96,152.17,179.50
8,acetaminophen 300 MG / codeine phosphate 30 MG Oral Tablet,15437,2843410.91,2371647.91,153.63,184.19
9,tizanidine 4 MG Oral Tablet,15515,2794153.49,2357349.42,151.94,180.09


In [12]:
# Drug spending by ingredient (rolled up)
ingredient_costs = con.execute("""
    SELECT 
        ingredient.concept_name as ingredient_name,
        COUNT(*) as rx_count,
        ROUND(SUM(cost.total_paid), 2) as total_paid,
        ROUND(AVG(cost.total_paid), 2) as avg_paid_per_rx,
        ROUND(SUM(cost.total_charge), 2) as total_charges,
        ROUND(100.0 * SUM(cost.total_paid) / SUM(cost.total_charge), 1) as pct_of_charges_paid
    FROM omop.drug_exposure de
    JOIN omop.cost cost 
        ON de.drug_exposure_id = cost.cost_event_id 
        AND cost.cost_domain_id = 'Drug'
    JOIN omop.concept_ancestor ca 
        ON de.drug_concept_id = ca.descendant_concept_id
    JOIN omop.concept ingredient 
        ON ca.ancestor_concept_id = ingredient.concept_id
    WHERE ingredient.vocabulary_id = 'RxNorm'
      AND ingredient.concept_class_id = 'Ingredient'
      AND de.drug_concept_id IS NOT NULL
      AND de.drug_concept_id != '0'
      AND cost.total_paid > 0
    GROUP BY ingredient.concept_name
    ORDER BY total_paid DESC
    LIMIT 15
""").fetchdf()

print("Top 15 Drug Ingredients by Total Spending")
print("=" * 100)
ingredient_costs

Top 15 Drug Ingredients by Total Spending


,ingredient_name,rx_count,total_paid,avg_paid_per_rx,total_charges,pct_of_charges_paid
0,acetaminophen,157059,23628680.94,150.44,27960039.83,84.5
1,hydrocodone,117884,17610714.74,149.39,20809877.83,84.6
2,tramadol,53935,8106882.08,150.31,9568682.84,84.7
3,cyclobenzaprine,53197,8005108.31,150.48,9380077.29,85.3
4,ibuprofen,44574,6689309.72,150.07,7865178.70,85.0
5,gabapentin,42968,6421216.15,149.44,7621255.74,84.3
6,naproxen,38460,5694687.26,148.07,6732011.70,84.6
7,meloxicam,29378,4463493.69,151.93,5261093.62,84.8
8,pregabalin,23235,3500712.58,150.67,4118227.17,85.0
9,ketorolac,21200,3224403.03,152.09,3803423.23,84.8


### NADAC Benchmark Comparison (National Average Drug Acquisition Cost)

Comparing workers' compensation drug costs to Medicaid NADAC national benchmarks from Mimilabs.

In [13]:
# Get top NDCs from our data for NADAC comparison
top_ndcs = con.execute("""
    SELECT 
        de.drug_source_value as ndc,
        c.concept_name as drug_name,
        COUNT(*) as rx_count,
        ROUND(AVG(cost.total_paid), 2) as wc_avg_paid,
        ROUND(AVG(de.quantity), 1) as avg_quantity
    FROM omop.drug_exposure de
    JOIN omop.cost cost 
        ON de.drug_exposure_id = cost.cost_event_id 
        AND cost.cost_domain_id = 'Drug'
    LEFT JOIN omop.concept c 
        ON de.drug_concept_id = c.concept_id
    WHERE de.drug_source_value IS NOT NULL
      AND LENGTH(de.drug_source_value) = 11  -- 11-digit NDCs only
      AND cost.total_paid > 0
      AND de.quantity > 0
    GROUP BY de.drug_source_value, c.concept_name
    HAVING COUNT(*) > 1000
    ORDER BY rx_count DESC
    LIMIT 20
""").fetchdf()

# Calculate cost per unit
top_ndcs['wc_cost_per_unit'] = round(top_ndcs['wc_avg_paid'] / top_ndcs['avg_quantity'], 3)

print("Top NDCs with Workers' Comp Cost Per Unit")
print("=" * 100)
print("These can be compared against NADAC national benchmarks")
top_ndcs[['ndc', 'drug_name', 'rx_count', 'wc_avg_paid', 'avg_quantity', 'wc_cost_per_unit']]

Top NDCs with Workers' Comp Cost Per Unit
These can be compared against NADAC national benchmarks


,ndc,drug_name,rx_count,wc_avg_paid,avg_quantity,wc_cost_per_unit
0,99999999999,No matching concept,15820,150.87,113.4,1.330
1,65162062711,tramadol hydrochloride 50 MG Oral Tablet,14600,148.07,161.2,0.919
2,53746046605,ibuprofen 800 MG Oral Tablet,10151,152.29,200.0,0.761
3,00591085305,acetaminophen 325 MG / hydrocodone bitartrate 10 MG Oral Tablet,8647,144.04,531.3,0.271
4,53746019005,naproxen 500 MG Oral Tablet,8517,151.17,79.1,1.911
5,00603388732,acetaminophen 325 MG / hydrocodone bitartrate 10 MG Oral Tablet,8329,151.01,156.7,0.964
6,68462019005,naproxen 500 MG Oral Tablet,7224,148.73,79.2,1.878
7,68382031910,tramadol hydrochloride 50 MG Oral Tablet,7103,151.24,87.1,1.736
8,00025152531,celecoxib 200 MG Oral Capsule [Celebrex],7017,145.53,243.4,0.598
9,59746017710,cyclobenzaprine hydrochloride 10 MG Oral Tablet,6995,150.12,133.5,1.124


## Procedure Cost Analysis

Analyzing workers' compensation procedure spending and comparing to Medicare benchmarks.

In [14]:
# Top procedures by total spending
procedure_costs = con.execute("""
    SELECT 
        po.procedure_source_value as hcpcs_code,
        c.concept_name as procedure_name,
        COUNT(*) as procedure_count,
        ROUND(SUM(cost.total_charge), 2) as total_charges,
        ROUND(SUM(cost.total_paid), 2) as total_paid,
        ROUND(AVG(cost.total_paid), 2) as wc_avg_paid,
        ROUND(AVG(cost.total_charge), 2) as avg_charge,
        ROUND(100.0 * SUM(cost.total_paid) / NULLIF(SUM(cost.total_charge), 0), 1) as pct_paid
    FROM omop.procedure_occurrence po
    JOIN omop.cost cost 
        ON po.procedure_occurrence_id = cost.cost_event_id 
        AND cost.cost_domain_id = 'Procedure'
    LEFT JOIN omop.concept c 
        ON po.procedure_concept_id = c.concept_id
    WHERE po.procedure_source_value IS NOT NULL
      AND cost.total_paid > 0
    GROUP BY po.procedure_source_value, c.concept_name
    ORDER BY total_paid DESC
    LIMIT 20
""").fetchdf()

print("Top 20 Procedures by Total Spending")
print("=" * 120)
procedure_costs

Top 20 Procedures by Total Spending


,hcpcs_code,procedure_name,procedure_count,total_charges,total_paid,wc_avg_paid,avg_charge,pct_paid
0,G0283,"Electrical stimulation (unattended), to one or more areas for indication(s) other than wound care, as part of a therapy plan of care",3672,2114466.35,1095824.14,298.43,575.84,51.8
1,99.99,Other miscellaneous procedures,969,738749.57,335543.52,346.28,762.38,45.4
2,J8499,"Prescription drug, oral, non chemotherapeutic, nos",204,126077.14,82123.24,402.56,618.03,65.1
3,J3490,Unclassified drugs,293,130925.54,74497.87,254.26,446.84,56.9
4,G0299,"Direct skilled nursing services of a registered nurse (rn) in the home health or hospice setting, each 15 minutes",188,122301.59,54554.26,290.18,650.54,44.6
5,J7799,"Noc drugs, other than inhalation drugs, administered through dme",104,91058.59,52288.13,502.77,875.56,57.4
6,86.59,Closure of skin and subcutaneous tissue of other sites,44,51482.73,42056.68,955.83,1170.06,81.7
7,J3590,Unclassified biologics,153,63533.45,35679.53,233.20,415.25,56.2
8,S9124,"Nursing care, in the home; by licensed practical nurse, per hour",149,56010.80,32024.55,214.93,375.91,57.2
9,S9123,"Nursing care, in the home; by registered nurse, per hour (use for general nursing care only, not to be used when cpt codes 99500-99602 can be used)",107,46001.35,28241.09,263.94,429.92,61.4


### Medicare Fee Schedule Benchmarks

Comparing workers' compensation procedure payments to Medicare national averages from Mimilabs (datacmsgov.mupphy).

| HCPCS | Description | Medicare Avg Allowed |
|-------|-------------|---------------------|
| G0283 | Electrical stimulation (unattended) | $10.24 |
| 97110 | Therapeutic exercise (15 min) | $24-26 |
| 97140 | Manual therapy (15 min) | $21-23 |
| 99213 | E&M established patient (15-29 min) | $66-83 |
| 99214 | E&M established patient (30-39 min) | $97-118 |

*Note: Workers' comp typically pays 100-200%+ of Medicare rates depending on state fee schedules.*

## Opioid Prescribing Deep Dive

Detailed analysis of opioid utilization patterns in workers' compensation claims. With 6.8M opioid agonist prescriptions, this represents the largest drug class in the dataset.

In [15]:
# Opioid prescriptions by specific drug
opioid_drugs = con.execute("""
    SELECT 
        ingredient.concept_name as opioid_ingredient,
        COUNT(*) as rx_count,
        ROUND(SUM(cost.total_paid), 2) as total_paid,
        ROUND(AVG(cost.total_paid), 2) as avg_paid,
        ROUND(AVG(de.days_supply), 1) as avg_days_supply,
        ROUND(AVG(de.quantity), 1) as avg_quantity
    FROM omop.drug_exposure de
    JOIN omop.concept_ancestor ca 
        ON de.drug_concept_id = ca.descendant_concept_id
    JOIN omop.concept ingredient 
        ON ca.ancestor_concept_id = ingredient.concept_id
    JOIN omop.concept c 
        ON de.drug_concept_id = c.concept_id
    JOIN reference_data.rxclass rc 
        ON c.concept_code = rc.rxcui
    LEFT JOIN omop.cost cost 
        ON de.drug_exposure_id = cost.cost_event_id 
        AND cost.cost_domain_id = 'Drug'
    WHERE rc.className = 'Opioid Agonist'
      AND rc.classType = 'EPC'
      AND ingredient.vocabulary_id = 'RxNorm'
      AND ingredient.concept_class_id = 'Ingredient'
    GROUP BY ingredient.concept_name
    ORDER BY rx_count DESC
    LIMIT 15
""").fetchdf()

print("Opioid Prescriptions by Ingredient")
print("=" * 100)
opioid_drugs

Opioid Prescriptions by Ingredient


,opioid_ingredient,rx_count,total_paid,avg_paid,avg_days_supply,avg_quantity
0,acetaminophen,4508806,38755588.28,142.01,21.5,235.8
1,hydrocodone,4056836,34750280.86,141.41,21.8,240.7
2,tramadol,1849514,15923621.34,142.39,19.1,152.6
3,oxycodone,460516,4061861.28,144.51,26.3,319.3
4,morphine,186192,1574521.48,140.18,29.7,194.4
5,fentanyl,98746,850417.44,139.28,28.9,51.1
6,hydromorphone,62048,533732.44,137.28,27.5,290.1
7,propoxyphene,54872,482947.80,148.23,16.9,367.0
8,methadone,45066,396671.18,141.87,30.1,568.0
9,ibuprofen,32958,274499.56,138.92,18.9,196.1


In [16]:
# Opioid prescribing trends over time
opioid_trends = con.execute("""
    SELECT 
        EXTRACT(YEAR FROM de.drug_exposure_start_date) as year,
        COUNT(*) as opioid_rx_count,
        COUNT(DISTINCT de.person_id) as unique_patients,
        ROUND(AVG(de.days_supply), 1) as avg_days_supply
    FROM omop.drug_exposure de
    JOIN omop.concept c 
        ON de.drug_concept_id = c.concept_id
    JOIN reference_data.rxclass rc 
        ON c.concept_code = rc.rxcui
    WHERE rc.className = 'Opioid Agonist'
      AND rc.classType = 'EPC'
      AND de.drug_exposure_start_date IS NOT NULL
      AND EXTRACT(YEAR FROM de.drug_exposure_start_date) BETWEEN 2015 AND 2025
    GROUP BY EXTRACT(YEAR FROM de.drug_exposure_start_date)
    ORDER BY year
""").fetchdf()

print("Opioid Prescribing Trends by Year")
print("=" * 80)
print("Tracking opioid utilization patterns over time")
opioid_trends

Opioid Prescribing Trends by Year
Tracking opioid utilization patterns over time


,year,opioid_rx_count,unique_patients,avg_days_supply
0,2015,470188,65499,25.3
1,2016,404584,55973,22.3
2,2017,346492,49317,22.6
3,2018,290242,42591,23.3
4,2019,240616,36898,22.6
5,2020,383694,27338,23.5
6,2021,164480,22816,23.8
7,2022,356,79,27.2
8,2023,117666,16620,23.8
9,2024,108732,15439,23.8


## Temporal Trends Analysis

Year-over-year patterns in workers' compensation healthcare utilization and spending.

In [17]:
# Overall utilization trends by year
utilization_trends = con.execute("""
    SELECT 
        year,
        SUM(conditions) as conditions,
        SUM(procedures) as procedures,
        SUM(drug_exposures) as drug_exposures,
        SUM(visits) as visits
    FROM (
        SELECT EXTRACT(YEAR FROM condition_start_date) as year, 
               COUNT(*) as conditions, 0 as procedures, 0 as drug_exposures, 0 as visits
        FROM omop.condition_occurrence 
        WHERE condition_start_date IS NOT NULL
        GROUP BY EXTRACT(YEAR FROM condition_start_date)
        
        UNION ALL
        
        SELECT EXTRACT(YEAR FROM procedure_date), 
               0, COUNT(*), 0, 0
        FROM omop.procedure_occurrence 
        WHERE procedure_date IS NOT NULL
        GROUP BY EXTRACT(YEAR FROM procedure_date)
        
        UNION ALL
        
        SELECT EXTRACT(YEAR FROM drug_exposure_start_date), 
               0, 0, COUNT(*), 0
        FROM omop.drug_exposure 
        WHERE drug_exposure_start_date IS NOT NULL
        GROUP BY EXTRACT(YEAR FROM drug_exposure_start_date)
        
        UNION ALL
        
        SELECT EXTRACT(YEAR FROM visit_start_date), 
               0, 0, 0, COUNT(*)
        FROM omop.visit_occurrence 
        WHERE visit_start_date IS NOT NULL
        GROUP BY EXTRACT(YEAR FROM visit_start_date)
    ) combined
    WHERE year BETWEEN 2015 AND 2025
    GROUP BY year
    ORDER BY year
""").fetchdf()

print("Healthcare Utilization Trends by Year")
print("=" * 80)
utilization_trends

Healthcare Utilization Trends by Year


,year,conditions,procedures,drug_exposures,visits
0,2015,5753826.0,202006.0,1196084.0,3672065.0
1,2016,4746769.0,177405.0,1137528.0,3062576.0
2,2017,4581260.0,159045.0,1007654.0,2869530.0
3,2018,4881714.0,152042.0,940283.0,2980104.0
4,2019,5177103.0,147685.0,871198.0,3074845.0
5,2020,4668269.0,51328.0,1192424.0,2587174.0
6,2021,4439835.0,38731.0,623260.0,2525938.0
7,2022,4567792.0,40192.0,146860.0,2581958.0
8,2023,4886604.0,85149.0,643687.0,2688901.0
9,2024,3907499.0,80986.0,640876.0,2210227.0


In [18]:
# Spending trends by year and domain
spending_trends = con.execute("""
    SELECT 
        EXTRACT(YEAR FROM de.drug_exposure_start_date) as year,
        'Drug' as domain,
        COUNT(*) as record_count,
        ROUND(SUM(cost.total_paid), 2) as total_paid,
        ROUND(AVG(cost.total_paid), 2) as avg_paid
    FROM omop.drug_exposure de
    JOIN omop.cost cost 
        ON de.drug_exposure_id = cost.cost_event_id 
        AND cost.cost_domain_id = 'Drug'
    WHERE de.drug_exposure_start_date IS NOT NULL
      AND EXTRACT(YEAR FROM de.drug_exposure_start_date) BETWEEN 2018 AND 2025
      AND cost.total_paid > 0
    GROUP BY EXTRACT(YEAR FROM de.drug_exposure_start_date)
    
    UNION ALL
    
    SELECT 
        EXTRACT(YEAR FROM po.procedure_date) as year,
        'Procedure' as domain,
        COUNT(*) as record_count,
        ROUND(SUM(cost.total_paid), 2) as total_paid,
        ROUND(AVG(cost.total_paid), 2) as avg_paid
    FROM omop.procedure_occurrence po
    JOIN omop.cost cost 
        ON po.procedure_occurrence_id = cost.cost_event_id 
        AND cost.cost_domain_id = 'Procedure'
    WHERE po.procedure_date IS NOT NULL
      AND EXTRACT(YEAR FROM po.procedure_date) BETWEEN 2018 AND 2025
      AND cost.total_paid > 0
    GROUP BY EXTRACT(YEAR FROM po.procedure_date)
    
    ORDER BY year, domain
""").fetchdf()

# Pivot the data for better display
spending_pivot = spending_trends.pivot(index='year', columns='domain', values=['total_paid', 'avg_paid'])
print("Spending Trends by Year and Domain")
print("=" * 80)
spending_pivot

Spending Trends by Year and Domain


total_paid            avg_paid          
domain         Drug  Procedure     Drug Procedure
year                                             
2018     8195503.00  112910.48   151.12    256.03
2019     7605124.18  144287.02   151.35    362.53
2020    10321369.07   35627.86   149.62    265.88
2021     5407919.35   24049.22   150.46    220.64
2022     1281566.92   25311.72   153.44    258.28
2023     5576610.38   56472.19   150.92    275.47
2024     5540363.82   68673.64   150.34    312.15
2025     4507355.52   50343.29   151.53    266.37

## Provider Concentration Analysis

Analyzing which providers drive the most utilization and spending in workers' compensation claims.

In [19]:
# Top providers by procedure volume and spending
top_providers = con.execute("""
    SELECT 
        p.provider_name,
        p.npi,
        p.specialty_source_value as specialty,
        COUNT(*) as procedure_count,
        ROUND(SUM(cost.total_paid), 2) as total_paid,
        ROUND(AVG(cost.total_paid), 2) as avg_paid_per_procedure
    FROM omop.procedure_occurrence po
    JOIN omop.provider p ON po.provider_id = p.provider_id
    LEFT JOIN omop.cost cost 
        ON po.procedure_occurrence_id = cost.cost_event_id 
        AND cost.cost_domain_id = 'Procedure'
    WHERE p.provider_id IS NOT NULL
      AND cost.total_paid > 0
    GROUP BY p.provider_name, p.npi, p.specialty_source_value
    ORDER BY total_paid DESC
    LIMIT 20
""").fetchdf()

print("Top 20 Providers by Procedure Spending")
print("=" * 120)
top_providers

Top 20 Providers by Procedure Spending


,provider_name,npi,specialty,procedure_count,total_paid,avg_paid_per_procedure
0,,None,202C00000X,2819,901796.40,319.90
1,"ATLIN, NEIL",1356370282,None,30,16194.74,539.82
2,"WOOD, LARRY",1669652111,None,33,14157.45,429.01
3,CTC TRINITY,None,None,1,11479.29,11479.29
4,"GARCIA, DANIEL",1255544433,None,1,10174.20,10174.20
5,"MOORE, ROBERT",1548208564,None,1,10000.00,10000.00
6,"FRISCE, TARYN",1831610294,None,5,9102.67,1820.53
7,"UNG, DAN",1346303351,None,24,8902.39,370.93
8,"GUTIERREZ, ALEXANDER",1013987999,None,4,8828.21,2207.05
9,"BARNES, GARY",1477821890,None,2,8347.39,4173.70


In [20]:
# Provider concentration analysis - what % of spending comes from top providers?
provider_concentration = con.execute("""
    WITH provider_spending AS (
        SELECT 
            p.provider_id,
            SUM(cost.total_paid) as total_paid,
            ROW_NUMBER() OVER (ORDER BY SUM(cost.total_paid) DESC) as spending_rank
        FROM omop.procedure_occurrence po
        JOIN omop.provider p ON po.provider_id = p.provider_id
        JOIN omop.cost cost 
            ON po.procedure_occurrence_id = cost.cost_event_id 
            AND cost.cost_domain_id = 'Procedure'
        WHERE cost.total_paid > 0
        GROUP BY p.provider_id
    ),
    total AS (
        SELECT SUM(total_paid) as grand_total FROM provider_spending
    )
    SELECT 
        CASE 
            WHEN spending_rank <= 10 THEN 'Top 10'
            WHEN spending_rank <= 50 THEN 'Top 11-50'
            WHEN spending_rank <= 100 THEN 'Top 51-100'
            WHEN spending_rank <= 500 THEN 'Top 101-500'
            ELSE 'Rest'
        END as provider_tier,
        COUNT(*) as provider_count,
        ROUND(SUM(ps.total_paid), 2) as tier_spending,
        ROUND(100.0 * SUM(ps.total_paid) / MAX(t.grand_total), 2) as pct_of_total
    FROM provider_spending ps
    CROSS JOIN total t
    GROUP BY CASE 
            WHEN spending_rank <= 10 THEN 'Top 10'
            WHEN spending_rank <= 50 THEN 'Top 11-50'
            WHEN spending_rank <= 100 THEN 'Top 51-100'
            WHEN spending_rank <= 500 THEN 'Top 101-500'
            ELSE 'Rest'
        END
    ORDER BY MIN(spending_rank)
""").fetchdf()

print("Provider Concentration Analysis")
print("=" * 80)
print("What percentage of procedure spending comes from top providers?")
provider_concentration

Provider Concentration Analysis
What percentage of procedure spending comes from top providers?


,provider_tier,provider_count,tier_spending,pct_of_total
0,Top 10,10,997271.84,44.74
1,Top 11-50,40,182584.37,8.19
2,Top 51-100,50,118487.93,5.32
3,Top 101-500,400,391534.28,17.56
4,Rest,2856,539241.42,24.19


## Key Findings Summary

### Clinical Profile
- **Musculoskeletal injuries dominate**: Lumbar sprains, low back pain, and neck sprains are the top conditions
- **Pain management focus**: The vast majority of drug utilization centers on pain relief

### Drug Utilization Concerns
- **Heavy opioid use**: Opioid agonists represent the #1 drug class with 6.8M prescriptions
- **Top opioids**: Hydrocodone and tramadol are the most prescribed
- **Benchmark opportunity**: Compare costs per unit against NADAC national averages

### Procedure Patterns  
- **Electrical stimulation (G0283)** is the top procedure by volume (1.3M occurrences)
- **Home nursing services** (S9123, S9124, G0299) represent significant utilization
- **Benchmark opportunity**: Compare payments to Medicare fee schedule rates

### Provider Concentration
- A small number of providers may drive a disproportionate share of spending
- Use NPI linkage to Mimilabs NPPES data for provider specialty verification

### Recommended Actions
1. **Cost benchmarking**: Query Mimilabs NADAC for drug price comparisons
2. **Fee schedule review**: Compare procedure payments to Medicare rates
3. **Opioid monitoring**: Track prescribing trends and identify outlier providers
4. **Provider profiling**: Cross-reference with NPPES for specialty validation

In [21]:
# Overall spending summary
total_spending = con.execute("""
    SELECT 
        cost_domain_id as domain,
        COUNT(*) as record_count,
        ROUND(SUM(total_charge), 2) as total_charges,
        ROUND(SUM(total_paid), 2) as total_paid,
        ROUND(AVG(total_paid), 2) as avg_paid
    FROM omop.cost
    WHERE total_paid > 0
    GROUP BY cost_domain_id
    ORDER BY total_paid DESC
""").fetchdf()

print("Total Spending Summary by Domain")
print("=" * 80)
total_spending

Total Spending Summary by Domain


,domain,record_count,total_charges,total_paid,avg_paid
0,Visit,8815909,2.991054e+11,7.017813e+10,7960.40
1,Procedure,33275191,2.141580e+10,9.437935e+09,283.63
2,Drug,9986430,1.956795e+09,1.456839e+09,145.88


In [22]:
# Close database connection
con.close()
print("Analysis complete. Database connection closed.")

Analysis complete. Database connection closed.
